In [ ]:
%%bash
set -euo pipefail

REPO_INPUT="/kaggle/input/datasets/uyendungthanh/medical-ie-round2-hybrid-v5-4-phobert"
LLAMA_INPUT="/kaggle/input/datasets/uyendungthanh/llama-cpp-colab-cuda-build-tar-gz"

if [ ! -d "$REPO_INPUT" ]; then
  echo "Không thấy repo input: $REPO_INPUT" >&2
  exit 1
fi
if [ ! -d "$LLAMA_INPUT" ]; then
  echo "Không thấy llama input: $LLAMA_INPUT" >&2
  exit 1
fi

if [ ! -d /kaggle/working/medical_ie_round2_hybrid_v5_4 ]; then
  cp -a "$REPO_INPUT"/. /kaggle/working/
else
  echo "Repo đã có trong Working — bỏ qua copy."
fi

mkdir -p /kaggle/working/medical_ie_local_assets
if [ ! -d /kaggle/working/medical_ie_local_assets/llama.cpp ]; then
  cp -a "$LLAMA_INPUT"/. /kaggle/working/medical_ie_local_assets/
else
  echo "llama.cpp đã có trong Working — bỏ qua copy."
fi

mkdir -p /kaggle/working/v5_1_runs
find /kaggle/working -maxdepth 3 -name VERSION -print
find /kaggle/working/medical_ie_local_assets -path '*/build/bin/llama-server' -print

### Khởi tạo đường dẫn

In [ ]:
import os
from pathlib import Path

repo_candidates = [
    Path("/kaggle/working/medical_ie_round2_hybrid_v5_4"),
    Path("/kaggle/working/medical_ie_round2_hybrid_v5_4/medical_ie_round2_hybrid_v5_4"),
]
REPO = next((p for p in repo_candidates if (p / "configs/hybrid_round2_v5_4_phobert_consensus.yaml").exists()), None)
if REPO is None:
    found = list(Path("/kaggle/working").glob("**/configs/hybrid_round2_v5_4_phobert_consensus.yaml"))
    if not found:
        raise FileNotFoundError("Không tìm thấy repo V5.4 trong /kaggle/working")
    REPO = found[0].parents[1]

RUN_ROOT = Path("/kaggle/working/v5_1_runs")
ASSET_DIR = Path("/kaggle/working/medical_ie_local_assets")
LLAMA_LIB_DIR = ASSET_DIR / "llama.cpp/build/bin"
LLAMA_BIN = LLAMA_LIB_DIR / "llama-server"
HF_HOME = Path("/kaggle/working/hf_cache")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)

for key, value in {
    "REPO": REPO,
    "RUN_ROOT": RUN_ROOT,
    "ASSET_DIR": ASSET_DIR,
    "LLAMA_LIB_DIR": LLAMA_LIB_DIR,
    "LLAMA_BIN": LLAMA_BIN,
    "HF_HOME": HF_HOME,
}.items():
    os.environ[key] = str(value)

os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["LD_LIBRARY_PATH"] = ":".join(dict.fromkeys([
    str(LLAMA_LIB_DIR),
    *[x for x in os.environ.get("LD_LIBRARY_PATH", "").split(":") if x],
]))

print("REPO:", REPO)
print("VERSION:", (REPO / "VERSION").read_text().strip())
print("RUN_ROOT:", RUN_ROOT)
print("LLAMA_BIN:", LLAMA_BIN)

In [ ]:
# %cd /kaggle/working/medical_ie_round2_hybrid_v4_multimodel
%cd $REPO

### Set Up dependency

In [ ]:
!apt-get update -qq
!apt-get install -y -qq git git-lfs cmake build-essential libcurl4-openssl-dev pigz zip
!git lfs install

!python -m pip install --no-cache-dir --upgrade pip wheel "setuptools<70" jedi
!python -m pip install --no-cache-dir seqeval==1.2.2 --no-build-isolation
!python -m pip uninstall -y gradio gradio-client
!python -m pip install --no-cache-dir -r requirements-v4.txt -r requirements-v5.txt
!python -m pip install --no-cache-dir "setuptools<70"
!python -m pip check || true

## 4. Khởi tạo lại môi trường sau Restart

In [1]:
import os
from pathlib import Path

found = list(Path("/kaggle/working").glob("**/configs/hybrid_round2_v5_4_phobert_consensus.yaml"))
if not found:
    raise FileNotFoundError("Không tìm thấy repo V5.4")
REPO = found[0].parents[1]
RUN_ROOT = Path("/kaggle/working/v5_1_runs")
ASSET_DIR = Path("/kaggle/working/medical_ie_local_assets")
LLAMA_LIB_DIR = ASSET_DIR / "llama.cpp/build/bin"
LLAMA_BIN = LLAMA_LIB_DIR / "llama-server"
HF_HOME = Path("/kaggle/working/hf_cache")

for key, value in {
    "REPO": REPO,
    "RUN_ROOT": RUN_ROOT,
    "ASSET_DIR": ASSET_DIR,
    "LLAMA_LIB_DIR": LLAMA_LIB_DIR,
    "LLAMA_BIN": LLAMA_BIN,
    "HF_HOME": HF_HOME,
}.items():
    os.environ[key] = str(value)

os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["LD_LIBRARY_PATH"] = ":".join(dict.fromkeys([
    str(LLAMA_LIB_DIR),
    *[x for x in os.environ.get("LD_LIBRARY_PATH", "").split(":") if x],
]))

print("REPO:", REPO)
print("VERSION:", (REPO / "VERSION").read_text().strip())

REPO: /kaggle/working/medical_ie_round2_hybrid_v5_4
VERSION: 5.4.0


In [2]:
%cd $REPO

/kaggle/working/medical_ie_round2_hybrid_v5_4


In [ ]:
# ktr conflic
!python -m pip check

In [3]:
# ktra môi trường cho buil lama
!git --version
!cmake --version | head -1
!gcc --version | head -1
!pigz --version

git version 2.34.1
cmake version 3.31.10
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
pigz 2.6


In [4]:
import sys, torch, transformers, sklearn, gliner
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("Transformers:", transformers.__version__)
print("scikit-learn:", sklearn.__version__)
print("GLiNER:", getattr(gliner, "__version__", "unknown"))
if sklearn.__version__ != "1.8.0":
    raise RuntimeError("Cần scikit-learn==1.8.0")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: True
GPU count: 2
Transformers: 4.57.6
scikit-learn: 1.8.0
GLiNER: 0.2.28


### Hàm check dung lượng

In [5]:
import shutil
from pathlib import Path

def disk_report(depth=2):
    usage = shutil.disk_usage("/kaggle/working")
    print(
        f"Working: used={usage.used/1024**3:.2f} GiB | "
        f"free={usage.free/1024**3:.2f} GiB | total={usage.total/1024**3:.2f} GiB"
    )
    roots = []
    for path in Path("/kaggle/working").iterdir():
        try:
            size = sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) if path.is_dir() else path.stat().st_size
            roots.append((size, path))
        except OSError:
            pass
    for size, path in sorted(roots, reverse=True)[:12]:
        print(f"{size/1024**3:7.2f} GiB  {path}")

def assert_free_gib(minimum):
    free = shutil.disk_usage("/kaggle/working").free / 1024**3
    print(f"Free: {free:.2f} GiB; required: {minimum:.2f} GiB")
    if free < minimum:
        raise RuntimeError("Không đủ dung lượng; dừng trước khi tạo checkpoint mới")

disk_report()

Working: used=0.21 GiB | free=19.29 GiB | total=19.52 GiB
   0.12 GiB  /kaggle/working/medical_ie_round2_hybrid_v5_4
   0.08 GiB  /kaggle/working/medical_ie_local_assets
   0.00 GiB  /kaggle/working/.virtual_documents
   0.00 GiB  /kaggle/working/v5_1_runs
   0.00 GiB  /kaggle/working/hf_cache


### LLAMA

In [6]:
%%bash
set -euo pipefail
LIB_DIR="$LLAMA_LIB_DIR"
cd "$LIB_DIR"
chmod +x llama-server

repair() {
  local lib="$1" major="$2" real="$3"
  [ -f "$real" ] || return 0
  file "$real" | grep -q ELF || { echo "File thật không hợp lệ: $real"; exit 1; }
  rm -f "$lib" "$major"
  ln -s "$real" "$major"
  ln -s "$major" "$lib"
  echo "repaired $lib -> $major -> $real"
}

repair libllama-common.so libllama-common.so.0 libllama-common.so.0.0.1
repair libllama.so        libllama.so.0        libllama.so.0.0.1
repair libmtmd.so         libmtmd.so.0         libmtmd.so.0.0.1
repair libggml.so         libggml.so.0         libggml.so.0.17.0
repair libggml-base.so    libggml-base.so.0    libggml-base.so.0.17.0
repair libggml-cpu.so     libggml-cpu.so.0     libggml-cpu.so.0.17.0
repair libggml-cuda.so    libggml-cuda.so.0    libggml-cuda.so.0.17.0

LD_LIBRARY_PATH="$LIB_DIR:${LD_LIBRARY_PATH:-}" ldd "$LLAMA_BIN" | grep -E 'not found|file too short' && exit 1 || true
LD_LIBRARY_PATH="$LIB_DIR:${LD_LIBRARY_PATH:-}" "$LLAMA_BIN" --version

repaired libllama-common.so -> libllama-common.so.0 -> libllama-common.so.0.0.1
repaired libllama.so -> libllama.so.0 -> libllama.so.0.0.1
repaired libmtmd.so -> libmtmd.so.0 -> libmtmd.so.0.0.1
repaired libggml.so -> libggml.so.0 -> libggml.so.0.17.0
repaired libggml-base.so -> libggml-base.so.0 -> libggml-base.so.0.17.0
repaired libggml-cpu.so -> libggml-cpu.so.0 -> libggml-cpu.so.0.17.0
repaired libggml-cuda.so -> libggml-cuda.so.0 -> libggml-cuda.so.0.17.0


version: 1 (da296d6)
built with GNU 11.4.0 for Linux x86_64


## 7. Xác minh lineage V5.1 và build knowledge

Verifier phải giữ nguyên byte-for-byte so với V5.1. Repo V5.2 đã sửa runner ở mức hạ tầng để nhận model alias; không sửa logic pipeline.


In [7]:
from pathlib import Path
import hashlib

verifier = Path("src/round2/llm_verifier.py")
digest = hashlib.sha256(verifier.read_bytes()).hexdigest()
expected = "206832a89ccefcb25c1beb81c80d103954c91d9258cc596398591f55f3b0ec58"
print("Verifier SHA-256:", digest)
assert digest == expected, "Qwen verifier không còn giống V5.1"
assert Path("configs/hybrid_round2_v5_4_phobert_consensus.yaml").is_file()
print("Lineage V5.1 verifier: OK")


Verifier SHA-256: 206832a89ccefcb25c1beb81c80d103954c91d9258cc596398591f55f3b0ec58
Lineage V5.1 verifier: OK


In [8]:
!python -m py_compile scripts/75_run_round2_local.py scripts/68_download_hybrid_models.py src/round2/llm_verifier.py

In [9]:
!python scripts/103_build_v5_knowledge.py

{'icd_rows': 17033, 'icd_aliases': 13425, 'rxnorm_rows': 68968}


In [10]:
!pytest -q

............................................................             [100%]
60 passed in 11.40s


In [11]:
!python -m compileall -q src scripts tests && echo 'compileall OK'

compileall OK


### VinhealthBERT

In [12]:
# đảm bảo gb
assert_free_gib(5.0)

Free: 19.29 GiB; required: 5.00 GiB


In [13]:
# chỉ tải base của vihealthbert trước
!python scripts/68_download_hybrid_models.py \
  --root "$REPO" \
  --skip-e5 \
  --skip-qwen

Fetching 6 files:   0%|                                   | 0/6 [00:00<?, ?it/s]
bpe.codes: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]


config.json: 100%|█████████████████████████████| 836/836 [00:00<00:00, 6.58MB/s]
vocab.txt: 895kB [00:00, 13.6MB/s]
bpe.codes: 1.14MB [00:00, 14.6MB/s]

pytorch_model.bin:   0%|                             | 0.00/540M [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]


.gitattributes: 1.22kB [00:00, 559kB/s]
README.md: 3.65kB [00:00, 4.33MB/s]
Fetching 6 files:  17%|████▌                      | 1/6 [00:00<00:00,  6.59it/s]
pytorch_model.bin:   0%|                             | 0.00/540M [00:00<?, ?B/s]
pytorch_model.bin:   0%|                             | 0.00/540M [00:00<?, ?B/s]
pytorch_model.bin:   0%|                             | 0.00/540M [00:00<?, ?B/s]
pytorch_model.bin:   0%|                             | 0.00/540M [00:00<?, ?B/s]
pytorch_model.bin:   0%|                             | 0.00/540M [00:01<?, ?B/s]
pytorch_model.bin: 

In [14]:
!find models/pretrained/vihealthbert-base-syllable -maxdepth 2 -type f \
  \( -name '*.safetensors' -o -name 'pytorch_model*.bin' \) -size +1M -print
!du -sh models/pretrained/vihealthbert-base-syllable

models/pretrained/vihealthbert-base-syllable/pytorch_model.bin
518M	models/pretrained/vihealthbert-base-syllable


In [15]:
# chuẩn bị
!python scripts/80_prepare_real100_manifest.py

{'real100': 100, 'folds': 5, 'output': '/kaggle/working/medical_ie_round2_hybrid_v5_4/data/processed/ner_real100'}


In [16]:
!python scripts/90_prepare_v4_training_data.py \
  --case-specs data/external_synthetic_v4/case_specs.jsonl \
  --marked-notes data/external_synthetic_v4/marked_notes.jsonl \
  --output-dir data/processed/v4_curriculum \
  --seed 42 \
  --mixed-synthetic-limit 150 \
  --real-repeat 3
!cat data/processed/v4_curriculum/stats.json

{'documents': 1458, 'entities': 9461, 'hard_negative_markers': 818, 'dropped_family': 0}
Saved /kaggle/working/medical_ie_round2_hybrid_v5_4/data/processed/v4_synthetic/manifest.jsonl
{
  "synthetic": {
    "documents": 1458,
    "entities": 9461,
    "hard_negative_markers": 818,
    "dropped_family": 0
  },
  "real_documents": 100,
  "real_train": 90,
  "real_dev": 10,
  "stage1_documents": 1458,
  "stage2_documents": 420,
  "stage2_real_occurrences": 270,
  "stage2_synthetic_occurrences": 150,
  "stage3_documents": 90,
  "policy": "synthetic warm-up -> real-dominant mixed -> real-only adaptation"
}
{
  "synthetic": {
    "documents": 1458,
    "entities": 9461,
    "hard_negative_markers": 818,
    "dropped_family": 0
  },
  "real_documents": 100,
  "real_train": 90,
  "real_dev": 10,
  "stage1_documents": 1458,
  "stage2_documents": 420,
  "stage2_real_occurrences": 270,
  "stage2_synthetic_occurrences": 150,
  "stage3_documents": 90,
  "policy": "synthetic warm-up -> real-dominant

In [ ]:
# stage 1
assert_free_gib(4.5)

In [17]:
!rm -rf models/ner/vihealthbert-round2-v4-curriculum-stage1
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python scripts/82_train_vihealthbert_real100.py \
  --config configs/ner_round2_v4_curriculum.yaml \
  --real-manifest data/processed/v4_curriculum/stage1_synthetic/manifest.jsonl \
  --pretrained-dir models/pretrained/vihealthbert-base-syllable \
  --output-dir models/ner/vihealthbert-round2-v4-curriculum-stage1 \
  --epochs 2 \
  --learning-rate 1.2e-5

{
  "window_preflight": {
    "requested_max_length": 256,
    "requested_stride": 64,
    "model_position_limit": 256,
    "effective_max_length": 256,
    "effective_stride": 64,
    "capped": false
  }
}
{
  "feature_preflight": {
    "feature_count": 1462,
    "token_count": 132241,
    "min_input_id": 0,
    "max_input_id": 63852,
    "min_label_id": 0,
    "max_label_id": 10
  }
}
Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at /kaggle/working/medical_ie_round2_hybrid_v5_4/models/pretrained/vihealthbert-base-syllable and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
{
  "embedding_preflight": {
    "tokenizer_required_vocab_size": 64001,
    "embedding_rows_before": 64001,
    "embedding_rows_after": 64001,
    "resized": false
  }
}
  0%|                                                    | 0/46 [00:00<?,

In [18]:
%%bash
set -euo pipefail
OUT="models/ner/vihealthbert-round2-v4-curriculum-stage1"
test -s "$OUT/training_summary.json"
WEIGHT="$(find "$OUT" -maxdepth 1 -type f \( -name '*.safetensors' -o -name 'pytorch_model*.bin' \) -size +1M -print -quit)"
test -n "$WEIGHT"
echo "Stage 1 valid: $WEIGHT"
find "$OUT" -maxdepth 1 -type d -name 'checkpoint-*' -exec rm -rf {} +
rm -rf models/pretrained/vihealthbert-base-syllable
sync
du -sh "$OUT"
df -h /kaggle/working

Stage 1 valid: models/ner/vihealthbert-round2-v4-curriculum-stage1/model.safetensors
515M	models/ner/vihealthbert-round2-v4-curriculum-stage1
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  732M   19G   4% /kaggle/working


In [19]:
# stage 2
!rm -rf models/ner/vihealthbert-round2-v4-curriculum-stage2
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python scripts/82_train_vihealthbert_real100.py \
  --config configs/ner_round2_v4_curriculum.yaml \
  --real-manifest data/processed/v4_curriculum/stage2_mixed/manifest.jsonl \
  --pretrained-dir models/ner/vihealthbert-round2-v4-curriculum-stage1 \
  --output-dir models/ner/vihealthbert-round2-v4-curriculum-stage2 \
  --epochs 4 \
  --learning-rate 8e-6

{
  "window_preflight": {
    "requested_max_length": 256,
    "requested_stride": 64,
    "model_position_limit": 256,
    "effective_max_length": 256,
    "effective_stride": 64,
    "capped": false
  }
}
{
  "feature_preflight": {
    "feature_count": 4260,
    "token_count": 1038735,
    "min_input_id": 0,
    "max_input_id": 63852,
    "min_label_id": 0,
    "max_label_id": 10
  }
}
{
  "embedding_preflight": {
    "tokenizer_required_vocab_size": 64001,
    "embedding_rows_before": 64001,
    "embedding_rows_after": 64001,
    "resized": false
  }
}
  0%|                                                   | 0/536 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
{'loss': 1.9788, 'grad_norm': 1.5721737146377563, 'learning_rate': 4.465116279069767e-06, 'e

In [20]:
%%bash
# check stage 2
set -euo pipefail
OUT="models/ner/vihealthbert-round2-v4-curriculum-stage2"
test -s "$OUT/training_summary.json"
WEIGHT="$(find "$OUT" -maxdepth 1 -type f \( -name '*.safetensors' -o -name 'pytorch_model*.bin' \) -size +1M -print -quit)"
test -n "$WEIGHT"
echo "Stage 2 valid: $WEIGHT"
find "$OUT" -maxdepth 1 -type d -name 'checkpoint-*' -exec rm -rf {} +
rm -rf models/ner/vihealthbert-round2-v4-curriculum-stage1
sync
du -sh "$OUT"
df -h /kaggle/working

Stage 2 valid: models/ner/vihealthbert-round2-v4-curriculum-stage2/model.safetensors
515M	models/ner/vihealthbert-round2-v4-curriculum-stage2
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  732M   19G   4% /kaggle/working


In [21]:
# stage 3
!rm -rf models/ner/vihealthbert-round2-v4-curriculum
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python scripts/82_train_vihealthbert_real100.py \
  --config configs/ner_round2_v4_curriculum.yaml \
  --real-manifest data/processed/v4_curriculum/stage3_real/manifest.jsonl \
  --pretrained-dir models/ner/vihealthbert-round2-v4-curriculum-stage2 \
  --output-dir models/ner/vihealthbert-round2-v4-curriculum \
  --epochs 5 \
  --learning-rate 5e-6

{
  "window_preflight": {
    "requested_max_length": 256,
    "requested_stride": 64,
    "model_position_limit": 256,
    "effective_max_length": 256,
    "effective_stride": 64,
    "capped": false
  }
}
{
  "feature_preflight": {
    "feature_count": 1370,
    "token_count": 341649,
    "min_input_id": 0,
    "max_input_id": 63852,
    "min_label_id": 0,
    "max_label_id": 10
  }
}
{
  "embedding_preflight": {
    "tokenizer_required_vocab_size": 64001,
    "embedding_rows_before": 64001,
    "embedding_rows_after": 64001,
    "resized": false
  }
}
  0%|                                                   | 0/172 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
{'loss': 0.5276, 'grad_norm': 1.6636645793914795, 'learning_rate': 4.683544303797468e-06, 'ep

In [22]:
%%bash
set -euo pipefail
FINAL="models/ner/vihealthbert-round2-v4-curriculum"
test -s "$FINAL/training_summary.json"
WEIGHT="$(find "$FINAL" -maxdepth 1 -type f \( -name '*.safetensors' -o -name 'pytorch_model*.bin' \) -size +1M -print -quit)"
test -n "$WEIGHT"
echo "Final ViHealthBERT valid: $WEIGHT"
find "$FINAL" -maxdepth 1 -type d -name 'checkpoint-*' -exec rm -rf {} +
rm -rf models/ner/vihealthbert-round2-v4-curriculum-stage2
sync
du -sh "$FINAL"
df -h /kaggle/working

Final ViHealthBERT valid: models/ner/vihealthbert-round2-v4-curriculum/model.safetensors
515M	models/ner/vihealthbert-round2-v4-curriculum
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  732M   19G   4% /kaggle/working


# PHẦN B — GLiNER V5.1 chunked training

In [ ]:
assert_free_gib(7.0)

In [23]:
# tải gliner
from huggingface_hub import snapshot_download
from pathlib import Path

dest = Path("models/gliner/gliner_multi-v2.1")
if not dest.exists() or not any(dest.rglob("*.safetensors")):
    snapshot_download(
        repo_id="urchade/gliner_multi-v2.1",
        local_dir=dest,
        local_dir_use_symlinks=False,
    )
print("GLiNER base:", dest)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

pytorch_model.bin:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

gliner_config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

GLiNER base: models/gliner/gliner_multi-v2.1


In [24]:
!du -sh models/gliner/gliner_multi-v2.1 && df -h /kaggle/working

2.2G	models/gliner/gliner_multi-v2.1
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.9G   17G  15% /kaggle/working


In [25]:
# data prepare
!rm -rf data/processed/gliner_v5_1_chunked_positive
!python scripts/92_prepare_gliner_v4_dataset.py \
  --train-manifest data/processed/v4_curriculum/stage2_mixed/manifest.jsonl \
  --dev-manifest data/processed/v4_curriculum/dev_real/manifest.jsonl \
  --output-dir data/processed/gliner_v5_1_chunked_positive \
  --chunk-size 220 \
  --overlap 64
!cat data/processed/gliner_v5_1_chunked_positive/chunk_stats.json

{
  "split": "train",
  "source_documents": 420,
  "long_documents": 270,
  "output_chunks": 3666,
  "dropped_empty_chunks": 99,
  "source_entities": 42651,
  "covered_unique_entities": 42651,
  "emitted_entities_with_overlap": 59982,
  "lost_entities": 0,
  "rescue_windows_added": 0,
  "chunk_size": 220,
  "overlap": 64,
  "max_output_tokens": 220,
  "positive_only": true,
  "label_counts_with_overlap": {
    "CHẨN_ĐOÁN": 9547,
    "TRIỆU_CHỨNG": 40421,
    "TÊN_XÉT_NGHIỆM": 5471,
    "KẾT_QUẢ_XÉT_NGHIỆM": 2576,
    "THUỐC": 1967
  }
}
{
  "split": "dev",
  "source_documents": 10,
  "long_documents": 10,
  "output_chunks": 140,
  "dropped_empty_chunks": 0,
  "source_entities": 1557,
  "covered_unique_entities": 1557,
  "emitted_entities_with_overlap": 2206,
  "lost_entities": 0,
  "rescue_windows_added": 0,
  "chunk_size": 220,
  "overlap": 64,
  "max_output_tokens": 220,
  "positive_only": true,
  "label_counts_with_overlap": {
    "CHẨN_ĐOÁN": 365,
    "TRIỆU_CHỨNG": 1511,
    "TÊN_

In [26]:
!python scripts/93_train_gliner_v4.py \
  --config configs/gliner_v5_2_train_batch4_1800.yaml \
  --validate-only

{
  "base": "/kaggle/working/medical_ie_round2_hybrid_v5_4/models/gliner/gliner_multi-v2.1",
  "output": "/kaggle/working/medical_ie_round2_hybrid_v5_4/models/gliner/medical-ie-v5-2",
  "local_files_only": false,
  "train_rows": 3666,
  "dev_rows": 140,
  "model_max_length": 256,
  "dataset_max_tokens": 220,
  "batch_size": 4,
  "eval_batch_size": 1,
  "gradient_accumulation_steps": 4,
  "effective_train_batch_size": 16,
  "fp16": true
}
{'status': 'validation_passed', 'config': '/kaggle/working/medical_ie_round2_hybrid_v5_4/configs/gliner_v5_2_train_batch4_1800.yaml'}


In [30]:
!pkill -f llama-server || true
!pkill -f 93_train_gliner_v4.py || true
!rm -rf models/gliner/medical-ie-v5-2
!nvidia-smi

Fri Jul 31 18:30:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P0             30W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [31]:
!CUDA_VISIBLE_DEVICES=0 \
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
TOKENIZERS_PARALLELISM=false \
OMP_NUM_THREADS=2 \
python scripts/93_train_gliner_v4.py \
  --config configs/gliner_v5_2_train_batch4_1800.yaml

{
  "base": "/kaggle/working/medical_ie_round2_hybrid_v5_4/models/gliner/gliner_multi-v2.1",
  "output": "/kaggle/working/medical_ie_round2_hybrid_v5_4/models/gliner/medical-ie-v5-2",
  "local_files_only": false,
  "train_rows": 3666,
  "dev_rows": 140,
  "model_max_length": 256,
  "dataset_max_tokens": 220,
  "batch_size": 4,
  "eval_batch_size": 1,
  "gradient_accumulation_steps": 4,
  "effective_train_batch_size": 16,
  "fp16": true
}
tokenizer_config.json: 100%|██████████████████| 52.0/52.0 [00:00<00:00, 293kB/s]
config.json: 100%|█████████████████████████████| 579/579 [00:00<00:00, 2.07MB/s]
spm.model: 100%|████████████████████████████| 4.31M/4.31M [00:06<00:00, 691kB/s]
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can p

In [32]:
%%bash
# xác minh tune 
set -euo pipefail
TUNED="models/gliner/medical-ie-v5-2"
test -s "$TUNED/training_summary.json"
WEIGHT="$(find "$TUNED" -type f \( -name '*.safetensors' -o -name 'pytorch_model*.bin' \) -not -path '*/checkpoint-*/*' -size +1M -print -quit)"
test -n "$WEIGHT"
echo "GLiNER tuned weight: $WEIGHT"

GLiNER tuned weight: models/gliner/medical-ie-v5-2/model.safetensors


In [33]:
%%bash
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 CUDA_VISIBLE_DEVICES='' python - <<'PY'
from gliner import GLiNER
model = GLiNER.from_pretrained('models/gliner/medical-ie-v5-2', local_files_only=True)
print('GLiNER tuned offline load: OK')
del model
PY

GLiNER tuned offline load: OK


In [34]:
%%bash
set -euo pipefail
TUNED="models/gliner/medical-ie-v5-2"
test -s "$TUNED/training_summary.json"
find "$TUNED" -type d -name 'checkpoint-*' -exec rm -rf {} +
rm -rf models/gliner/gliner_multi-v2.1
sync
du -sh "$TUNED"
df -h /kaggle/working

2.2G	models/gliner/medical-ie-v5-2
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.9G   17G  15% /kaggle/working


# PHẦN C — Assertion, BGE-M3, reranker và indexes

In [35]:
# assertion scope
!rm -f models/assertion_scope/v4_scope.joblib
!python scripts/94_prepare_assertion_scope_data.py
!python scripts/95_train_assertion_scope_classifier.py \
  --input data/processed/assertion_scope_v4.jsonl \
  --output models/assertion_scope/v4_scope.joblib

{'examples': 19774, 'output': '/kaggle/working/medical_ie_round2_hybrid_v5_4/data/processed/assertion_scope_v4.jsonl'}
{'train': 15819, 'eval': 3955, 'micro_f1': 0.7821435918535281, 'output': '/kaggle/working/medical_ie_round2_hybrid_v5_4/models/assertion_scope/v4_scope.joblib'}


In [36]:
%%bash
python - <<'PY'
import joblib, sklearn
p='models/assertion_scope/v4_scope.joblib'
a=joblib.load(p)
print('sklearn', sklearn.__version__)
print('labels', a['labels'])
print('micro_f1', a.get('micro_f1'))
print('assertion artifact OK')
PY

sklearn 1.8.0
labels ['isNegated', 'isHistorical', 'isFamily', 'isReporter', 'isUncertain']
micro_f1 0.7821435918535281
assertion artifact OK


## 18. Tải BGE-M3 và reranker, bỏ ONNX/OpenVINO

Các thư mục ONNX của BGE-M3 từng chiếm khoảng 2.2 GB nhưng pipeline này dùng PyTorch/Safetensors.

In [37]:
from huggingface_hub import snapshot_download
from pathlib import Path

ignore = [
    "onnx/*", "onnx/**", "openvino/*", "openvino/**",
    "*.onnx", "*.onnx_data", "*.xml", "*.blob",
]
models = [
    ("BAAI/bge-m3", Path("models/embeddings/bge-m3")),
    ("BAAI/bge-reranker-v2-m3", Path("models/rerankers/bge-reranker-v2-m3")),
]
for repo_id, dest in models:
    if not dest.exists() or not any(dest.rglob("*.safetensors")):
        snapshot_download(
            repo_id=repo_id,
            local_dir=dest,
            local_dir_use_symlinks=False,
            ignore_patterns=ignore,
        )
    print(repo_id, "->", dest)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

.DS_Store:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

BAAI/bge-reranker-v2-m3 -> models/rerankers/bge-reranker-v2-m3


In [38]:
!du -sh models/embeddings/bge-m3 models/rerankers/bge-reranker-v2-m3
!find models/embeddings/bge-m3 -maxdepth 2 -type d -name onnx -print
!df -h /kaggle/working

2.2G	models/embeddings/bge-m3
2.2G	models/rerankers/bge-reranker-v2-m3
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  7.2G   13G  37% /kaggle/working


## 19. Build ICD/RxNorm embedding indexes

In [39]:
!pkill -f llama-server || true
!python scripts/97_build_v4_embedding_indexes.py \
  --config configs/hybrid_round2_v5_4_phobert_consensus.yaml

Embedding model: models/embeddings/bge-m3
Batches: 100%|████████████████████████████████| 167/167 [01:11<00:00,  2.35it/s]
Saved ICD index: /kaggle/working/medical_ie_round2_hybrid_v5_4/data/processed/icd_embeddings_bge_m3.npz (10633 rows)
Batches: 100%|████████████████████████████████| 665/665 [03:29<00:00,  3.17it/s]
Saved RxNorm index: /kaggle/working/medical_ie_round2_hybrid_v5_4/data/processed/rxnorm_embeddings_bge_m3.npz (42550 rows)


In [40]:
!ls -lh data/processed/icd_embeddings_bge_m3.npz data/processed/rxnorm_embeddings_bge_m3.npz
!df -h /kaggle/working

-rw-r--r-- 1 root root  39M Jul 31 19:21 data/processed/icd_embeddings_bge_m3.npz
-rw-r--r-- 1 root root 156M Jul 31 19:24 data/processed/rxnorm_embeddings_bge_m3.npz
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  7.4G   13G  38% /kaggle/working


# PHẦN C2 — PhoBERT-base-v2 độc lập cho V5.4

PhoBERT là NER model thứ ba, độc lập với ViHealthBERT và GLiNER. Primary config chỉ nhận PhoBERT entity có exact corroboration để tránh union recall không kiểm soát.


In [41]:
# Tải PhoBERT-base-v2 vào local model store.
from huggingface_hub import snapshot_download
PHOBERT_BASE=REPO/'models/pretrained/phobert-base-v2'
if not PHOBERT_BASE.exists():
 snapshot_download(repo_id='vinai/phobert-base-v2',local_dir=str(PHOBERT_BASE),local_dir_use_symlinks=False)
assert (PHOBERT_BASE/'config.json').is_file();print(PHOBERT_BASE)


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

/kaggle/working/medical_ie_round2_hybrid_v5_4/models/pretrained/phobert-base-v2


In [43]:
# # Curriculum dùng lại dữ liệu đã chuẩn bị ở Phần A: synthetic -> real-dominant mixed -> real-only.
# assert_free_gib(5.0)
# ph_env={**env,'CUDA_VISIBLE_DEVICES':'0','PYTHONPATH':str(REPO),'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
# subprocess.run(['python','-u','scripts/110_train_phobert_v54_curriculum.py','--config','configs/ner_round2_v5_4_phobert.yaml'],cwd=REPO,env=ph_env,check=True)

import os
import subprocess

# Curriculum dùng lại dữ liệu đã chuẩn bị ở Phần A: synthetic -> real-dominant mixed -> real-only.
assert_free_gib(5.0)

# Thay 'env' bằng os.environ.copy() hoặc os.environ
ph_env = {
    **os.environ,
    'CUDA_VISIBLE_DEVICES': '0',
    'PYTHONPATH': str(REPO),
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'
}

subprocess.run(
    ['python', '-u', 'scripts/110_train_phobert_v54_curriculum.py', '--config', 'configs/ner_round2_v5_4_phobert.yaml'],
    cwd=REPO,
    env=ph_env,
    check=True
)

Free: 11.64 GiB; required: 5.00 GiB

===== PhoBERT synthetic warm-up =====
{
  "window_preflight": {
    "requested_max_length": 256,
    "requested_stride": 64,
    "model_position_limit": 256,
    "effective_max_length": 256,
    "effective_stride": 64,
    "capped": false
  }
}
{
  "feature_preflight": {
    "feature_count": 1462,
    "token_count": 132241,
    "min_input_id": 0,
    "max_input_id": 63852,
    "min_label_id": 0,
    "max_label_id": 10
  }
}


Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at /kaggle/working/medical_ie_round2_hybrid_v5_4/models/pretrained/phobert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{
  "embedding_preflight": {
    "tokenizer_required_vocab_size": 64001,
    "embedding_rows_before": 64001,
    "embedding_rows_after": 64001,
    "resized": false
  }
}


 27%|██▋       | 25/92 [00:06<00:15,  4.44it/s]

{'loss': 2.1373, 'grad_norm': 1.96933114528656, 'learning_rate': 9.714285714285715e-06, 'epoch': 0.27}


...

 97%|█████████▋| 250/258 [01:21<00:02,  3.41it/s]

{'loss': 0.612, 'grad_norm': 4.927876949310303, 'learning_rate': 1.89873417721519e-07, 'epoch': 2.91}


100%|██████████| 258/258 [01:27<00:00,  2.94it/s]


{'train_runtime': 87.6384, 'train_samples_per_second': 46.897, 'train_steps_per_second': 2.944, 'train_loss': 0.6308932747951773, 'epoch': 3.0}
{
  "train_metrics": {
    "train_runtime": 87.6384,
    "train_samples_per_second": 46.897,
    "train_steps_per_second": 2.944,
    "total_flos": 537008527088640.0,
    "train_loss": 0.6308932747951773,
    "epoch": 3.0
  },
  "source_documents": {
    "real_part2": 90
  },
  "feature_windows": 1370,
  "feature_preflight": {
    "token_count": 341649,
    "min_input_id": 0,
    "max_input_id": 63852,
    "min_label_id": 0,
    "max_label_id": 10
  },
  "embedding_preflight": {
    "tokenizer_required_vocab_size": 64001,
    "embedding_rows_before": 64001,
    "embedding_rows_after": 64001,
    "resized": false
  },
  "window_preflight": {
    "requested_max_length": 256,
    "requested_stride": 64,
    "model_position_limit": 256,
    "effective_max_length": 256,
    "effective_stride": 64,
    "capped": false
  },
  "config": "configs/ner_ro

CompletedProcess(args=['python', '-u', 'scripts/110_train_phobert_v54_curriculum.py', '--config', 'configs/ner_round2_v5_4_phobert.yaml'], returncode=0)

In [44]:
# Xác minh final PhoBERT; stage1/stage2 đã tự dọn để tiết kiệm disk.
PHOBERT_FINAL=REPO/'models/ner/phobert-round2-v5-4-curriculum'
assert (PHOBERT_FINAL/'config.json').is_file()
assert list(PHOBERT_FINAL.glob('*.safetensors')) or (PHOBERT_FINAL/'pytorch_model.bin').is_file()
assert (PHOBERT_FINAL/'training_summary.json').is_file()
print((PHOBERT_FINAL/'training_summary.json').read_text(encoding='utf-8')[:3000]);assert_free_gib(3.5)


{
  "train_metrics": {
    "train_runtime": 87.6384,
    "train_samples_per_second": 46.897,
    "train_steps_per_second": 2.944,
    "total_flos": 537008527088640.0,
    "train_loss": 0.6308932747951773,
    "epoch": 3.0
  },
  "source_documents": {
    "real_part2": 90
  },
  "feature_windows": 1370,
  "feature_preflight": {
    "token_count": 341649,
    "min_input_id": 0,
    "max_input_id": 63852,
    "min_label_id": 0,
    "max_label_id": 10
  },
  "embedding_preflight": {
    "tokenizer_required_vocab_size": 64001,
    "embedding_rows_before": 64001,
    "embedding_rows_after": 64001,
    "resized": false
  },
  "window_preflight": {
    "requested_max_length": 256,
    "requested_stride": 64,
    "model_position_limit": 256,
    "effective_max_length": 256,
    "effective_stride": 64,
    "capped": false
  },
  "config": "configs/ner_round2_v5_4_phobert.yaml",
  "real_manifest": "data/processed/v4_curriculum/stage3_real/manifest.jsonl",
  "extra_manifest": null
}
Free: 9.63 GiB

# PHẦN D — Qwen, kiểm tra asset và inference

In [45]:
assert_free_gib(3.5)

Free: 9.63 GiB; required: 3.50 GiB


In [46]:
# !python scripts/68_download_hybrid_models.py \
#   --root "$REPO" \
#   --skip-vihealthbert \
#   --skip-e5
# !ls -lh models/gguf/qwen3-4b/Qwen3-4B-Q4_K_M.gguf

assert_free_gib(5.5)
!python scripts/68_download_hybrid_models.py \
  --root "$REPO" \
  --skip-vihealthbert \
  --skip-e5 \
  --qwen-repo bartowski/Qwen2.5-7B-Instruct-GGUF \
  --qwen-file Qwen2.5-7B-Instruct-Q4_K_M.gguf \
  --qwen-dir models/gguf/qwen2.5-7b
!ls -lh models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf


Free: 9.63 GiB; required: 5.50 GiB
Qwen2.5-7B-Instruct-Q4_K_M.gguf: 100%|██████| 4.68G/4.68G [00:32<00:00, 145MB/s]
Qwen GGUF downloaded: /kaggle/working/medical_ie_round2_hybrid_v5_4/models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf
Manifest: /kaggle/working/medical_ie_round2_hybrid_v5_4/models/local_model_manifest.json
Download completed. You can now build indexes and pack the model store.
-rw-r--r-- 1 root root 4.4G Jul 31 19:36 models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf


## 21. Kiểm tra inference assets

Không dùng `--mode all` vì base ViHealthBERT/GLiNER đã chủ động xóa sau khi fine-tune.

In [47]:
!python scripts/102_check_v4_assets.py \
  --root "$REPO" \
  --mode inference

[OK] ViHealthBERT V4: /kaggle/working/medical_ie_round2_hybrid_v5_4/models/ner/vihealthbert-round2-v4-curriculum
[OK] GLiNER V5.2: /kaggle/working/medical_ie_round2_hybrid_v5_4/models/gliner/medical-ie-v5-2
[OK] Assertion scope: /kaggle/working/medical_ie_round2_hybrid_v5_4/models/assertion_scope/v4_scope.joblib
[OK] ICD BGE index: /kaggle/working/medical_ie_round2_hybrid_v5_4/data/processed/icd_embeddings_bge_m3.npz
[OK] RxNorm BGE index: /kaggle/working/medical_ie_round2_hybrid_v5_4/data/processed/rxnorm_embeddings_bge_m3.npz
V5 asset check passed.


In [48]:
%%bash
set -euo pipefail
test -x "$LLAMA_BIN"
test -s models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf
test -d models/embeddings/bge-m3
test -d models/rerankers/bge-reranker-v2-m3
test -s data/processed/icd10_byt_v5.csv
test -s data/external/rxnorm_lookup_flat.json
LD_LIBRARY_PATH="$LLAMA_LIB_DIR:${LD_LIBRARY_PATH:-}" "$LLAMA_BIN" --version
echo "All runtime assets OK"

All runtime assets OK


version: 1 (da296d6)
built with GNU 11.4.0 for Linux x86_64


In [49]:
!grep -n -A6 '^vihealthbert:' configs/hybrid_round2_v5_context.yaml
!grep -n -A20 '^gliner:' configs/hybrid_round2_v5_context.yaml
!grep -n -A6 '^reranker:' configs/hybrid_round2_v5_context.yaml
!df -h /kaggle/working

80:vihealthbert:
81-  config_path: configs/ner_round2_v4_curriculum.yaml
82-  thresholds:
83-    TRIỆU_CHỨNG: 0.9
84-    CHẨN_ĐOÁN: 0.94
85-    TÊN_XÉT_NGHIỆM: 0.95
86-    KẾT_QUẢ_XÉT_NGHIỆM: 0.95
133:gliner:
134-  model_path: models/gliner/medical-ie-v4
135-  raw_threshold: 0.45
136-  device: auto
137-  labels:
138-  - triệu chứng
139-  - chẩn đoán
140-  - tên xét nghiệm
141-  - kết quả xét nghiệm
142-  - thuốc
143-  thresholds:
144-    TRIỆU_CHỨNG: 0.8
145-    CHẨN_ĐOÁN: 0.84
146-    TÊN_XÉT_NGHIỆM: 0.8
147-    KẾT_QUẢ_XÉT_NGHIỆM: 0.82
148-    THUỐC: 0.84
149-  uncorroborated_types: *id001
150-  overlap_iou: 0.4
151-  require_exact_support: false
152-  max_chars_by_type: *id002
153-  max_words_by_type: *id003
181:reranker:
182-  enabled: true
183-  model_path: models/rerankers/bge-reranker-v2-m3
184-  threshold: 0.57
185-  margin: 0.08
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   15G  5.3G  73% /kaggle/working


## 22. Audit parser V5-2

In [50]:
!python scripts/98_audit_round2_v4_structure.py \
  --input-dir data/raw/input_round_2 \
  --output "$RUN_ROOT/parser_audit_v5_1.json"

{
  "files": 100,
  "section_counts": [
    [
      "PAST_HISTORY",
      86
    ],
    [
      "CURRENT_ILLNESS",
      62
    ],
    [
      "UNKNOWN",
      56
    ],
    [
      "ADMISSION_REASON",
      53
    ],
    [
      "HOSPITAL_ASSESSMENT",
      49
    ],
    [
      "DOCTOR_ANSWER",
      49
    ],
    [
      "USER_QUESTION",
      43
    ],
    [
      "CURRENT_SYMPTOMS",
      35
    ],
    [
      "PRE_ADMISSION_EVENTS",
      35
    ],
    [
      "IMAGING",
      35
    ],
    [
      "ONSET",
      31
    ],
    [
      "PAST_MEDICATION",
      27
    ],
    [
      "SYMPTOM_CHARACTERISTICS",
      25
    ],
    [
      "DIAGNOSIS",
      25
    ],
    [
      "DISEASE_COURSE",
      22
    ],
    [
      "LAB_RESULT",
      18
    ],
    [
      "FAQ_SYMPTOMS",
      17
    ],
    [
      "PHYSICAL_EXAM",
      17
    ],
    [
      "TREATMENT",
      17
    ],
    [
      "PAST_PROCEDURE",
      17
    ],
    [
      "LAB_RESULT_EXTRA",
      12
    ],
    [
    

## 24. Chạy 100 file với resume/retry

Không dùng `--overwrite`: nếu server lỗi giữa chừng, lần thử sau bỏ qua file đã có đủ prediction và debug.

In [62]:
%%bash
set +e
STATUS=1
for ATTEMPT in 1 2 3 4; do
  echo "===== ATTEMPT $ATTEMPT ====="
  pkill -f 'llama-server.*8081' 2>/dev/null || true
  sleep 3

  LD_LIBRARY_PATH="$LLAMA_LIB_DIR:${LD_LIBRARY_PATH:-}" \
  PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  TOKENIZERS_PARALLELISM=false \
  python scripts/75_run_round2_local.py \
    --llama-bin "$LLAMA_BIN" \
    --qwen-model models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf \
    --config configs/hybrid_round2_v5_3_augmented_gliner.yaml \
    --input-dir data/raw/input_round_2 \
    --baseline-prediction-dir data/baselines/vihealthbert_27_6152 \
    --output-dir "$RUN_ROOT/full_hybrid/predictions_augment" \
    --debug-dir "$RUN_ROOT/full_hybrid/debug_augment" \
    --limit 100 \
    --gpu-layers 99 \
    --threads 4

  STATUS=$?
  [ $STATUS -eq 0 ] && break
  sleep 5
done

exit $STATUS

===== ATTEMPT 1 =====
[1/100] 1.txt: base=16 rule_raw=44 rule=38 vh_raw=82 vh=8 ph_raw=0 ph=0 gl_raw=0 gl=0 llm_raw=25 llm=24 merged=39 elapsed=68.5s
[2/100] 2.txt: base=28 rule_raw=16 rule=11 vh_raw=76 vh=18 ph_raw=0 ph=0 gl_raw=0 gl=0 llm_raw=31 llm=30 merged=35 elapsed=113.9s
[3/100] 3.txt: base=61 rule_raw=32 rule=27 vh_raw=99 vh=27 ph_raw=0 ph=0 gl_raw=0 gl=0 llm_raw=53 llm=51 merged=69 elapsed=185.3s
[4/100] 4.txt: base=62 rule_raw=34 rule=32 vh_raw=99 vh=42 ph_raw=0 ph=0 gl_raw=0 gl=0 llm_raw=44 llm=43 merged=67 elapsed=249.0s
[5/100] 5.txt: base=35 rule_raw=16 rule=16 vh_raw=81 vh=20 ph_raw=0 ph=0 gl_raw=0 gl=0 llm_raw=23 llm=23 merged=35 elapsed=288.4s
[6/100] 6.txt: base=66 rule_raw=32 rule=31 vh_raw=116 vh=37 ph_raw=0 ph=0 gl_raw=0 gl=0 llm_raw=44 llm=42 merged=70 elapsed=374.0s
[7/100] 7.txt: base=27 rule_raw=20 rule=19 vh_raw=58 vh=10 ph_raw=0 ph=0 gl_raw=0 gl=0 llm_raw=13 llm=13 merged=29 elapsed=429.8s
[8/100] 8.txt: base=51 rule_raw=26 rule=15 vh_raw=86 vh=27 ph_raw=0 p

/kaggle/working/medical_ie_round2_hybrid_v5_4/src/round2/factory.py:53: UserWarning: GLiNER model not found: /kaggle/working/medical_ie_round2_hybrid_v5_4/models/gliner/medical-ie-v5-3-augmented GLiNER disabled.
  except (FileNotFoundError,ImportError) as exc:warnings.warn(str(exc)+' GLiNER disabled.')
Chunks: 100%|██████████| 2/2 [00:00<00:00, 13.13it/s]
/usr/lib/python3.12/multiprocessing/resource_tracker.py:279: UserWarning: resource_tracker: There appear to be 2 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '


In [52]:
from pathlib import Path
import os

run_root = Path(os.environ["RUN_ROOT"]) / "full_hybrid"
pred = list((run_root / "predictions").glob("*.json"))
debug = list((run_root / "debug").glob("*.debug.json"))
print("predictions:", len(pred))
print("debug:", len(debug))
assert len(pred) == 100, "Chưa đủ 100 prediction"
assert len(debug) == 100, "Chưa đủ 100 debug"
print("✅ Đủ 100/100")

predictions: 100
debug: 100
✅ Đủ 100/100


In [63]:
%%bash
set -euo pipefail
cd "$RUN_ROOT/full_hybrid"
rm -f /kaggle/working/v5_4_predictions_augment.zip /kaggle/working/v5_4_debug_augment.zip
zip -r -1 /kaggle/working/v5_4_predictions_augment.zip predictions_augment
zip -r -1 /kaggle/working/v5_4_debug_augment.zip debug_augment
ls -lh /kaggle/working/v5_4_predictions_augment.zip /kaggle/working/v5_4_debug_augment.zip

  adding: predictions_augment/ (stored 0%)
  adding: predictions_augment/39.json (deflated 82%)
  adding: predictions_augment/79.json (deflated 77%)
  adding: predictions_augment/14.json (deflated 80%)
  adding: predictions_augment/45.json (deflated 79%)
  adding: predictions_augment/40.json (deflated 84%)
  adding: predictions_augment/26.json (deflated 83%)
  adding: predictions_augment/11.json (deflated 84%)
  adding: predictions_augment/5.json (deflated 83%)
  adding: predictions_augment/22.json (deflated 84%)
  adding: predictions_augment/6.json (deflated 84%)
  adding: predictions_augment/95.json (deflated 77%)
  adding: predictions_augment/15.json (deflated 82%)
  adding: predictions_augment/54.json (deflated 80%)
  adding: predictions_augment/72.json (deflated 80%)
  adding: predictions_augment/76.json (deflated 68%)
  adding: predictions_augment/92.json (deflated 81%)
  adding: predictions_augment/60.json (deflated 82%)
  adding: predictions_augment/10.json (deflated 85%)
  add

### BÁO CÁO DUNG LƯỢNG CUỐI

In [54]:
disk_report()

Working: used=14.28 GiB | free=5.22 GiB | total=19.52 GiB
  14.14 GiB  /kaggle/working/medical_ie_round2_hybrid_v5_4
   0.22 GiB  /kaggle/working/medical_ie_local_assets
   0.04 GiB  /kaggle/working/v5_1_runs
   0.01 GiB  /kaggle/working/hf_cache
   0.00 GiB  /kaggle/working/v5_4_debug_pho_consen.zip
   0.00 GiB  /kaggle/working/.virtual_documents
   0.00 GiB  /kaggle/working/v5_4_predictions_pho_consen.zip


In [55]:
!du -sh \
  models/ner/vihealthbert-round2-v4-curriculum \
  models/gliner/medical-ie-v5-2 \
  models/embeddings/bge-m3 \
  models/rerankers/bge-reranker-v2-m3 \
  models/gguf/qwen3-4b \
  data/processed/*bge_m3.npz \
  "$RUN_ROOT/full_hybrid" 2>/dev/null

515M	models/ner/vihealthbert-round2-v4-curriculum
2.2G	models/gliner/medical-ie-v5-2
2.2G	models/embeddings/bge-m3
2.2G	models/rerankers/bge-reranker-v2-m3
39M	data/processed/icd_embeddings_bge_m3.npz
156M	data/processed/rxnorm_embeddings_bge_m3.npz
43M	/kaggle/working/v5_1_runs/full_hybrid


### Xác Minh CLEAN UP

In [56]:
from src.common.schema import Entity
from src.postprocessing.precision_cleanup_v53 import PrecisionCleanupV53, PrecisionCleanupV53Config

def _e(text, typ, source='gliner'):
    return Entity(text=text,start=0,end=len(text),type=typ,source=source,confidence=.99)
cleanup=PrecisionCleanupV53(PrecisionCleanupV53Config())
rows,report=cleanup.apply([_e('Xông khí dung','THUỐC'),_e('Xông khí dung','THUỐC','baseline_anchor'),_e('sỏi mật','TRIỆU_CHỨNG')])
assert any(e.source=='baseline_anchor' for e in rows)
assert any(e.text=='sỏi mật' and e.type=='CHẨN_ĐOÁN' for e in rows)
assert not any(e.text=='Xông khí dung' and e.source=='gliner' for e in rows)
print('V5.3 cleanup regression: PASSED')


V5.3 cleanup regression: PASSED


# PHẦN F — Full-100 ablations

Chạy tuần tự, không chạy song song. `no_cleanup_control` phải gần tái lập kiến trúc score 31.1437; primary chỉ khác cleanup hẹp. Selective verifier chỉ kiểm Qwen/ViHealth và bypass GLiNER.

In [57]:
import os,subprocess,time,shutil,json
from pathlib import Path
from IPython.display import FileLink,display

ABLATION_ROOT=Path('/kaggle/working/v5_3_ablations');ABLATION_ROOT.mkdir(parents=True,exist_ok=True)
ABLATIONS={
 'no_cleanup_control':'configs/hybrid_round2_v5_3_no_cleanup_control.yaml',
 'primary_cleanup':'configs/hybrid_round2_v5_4_phobert_consensus.yaml',
 'type_repair_only':'configs/hybrid_round2_v5_3_type_repair_only.yaml',
 'selective_verifier':'configs/hybrid_round2_v5_3_selective_verifier.yaml',
}
env=os.environ.copy();env.update({'PYTHONUNBUFFERED':'1','PYTHONPATH':str(REPO),'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','HF_DATASETS_OFFLINE':'1','TOKENIZERS_PARALLELISM':'false','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'})
env['LD_LIBRARY_PATH']=f"{LLAMA_LIB_DIR}:{env.get('LD_LIBRARY_PATH','')}"
print(ABLATIONS)


{'no_cleanup_control': 'configs/hybrid_round2_v5_3_no_cleanup_control.yaml', 'primary_cleanup': 'configs/hybrid_round2_v5_4_phobert_consensus.yaml', 'type_repair_only': 'configs/hybrid_round2_v5_3_type_repair_only.yaml', 'selective_verifier': 'configs/hybrid_round2_v5_3_selective_verifier.yaml'}


In [ ]:
# tái hiện
def run_v53(name,config,attempts=4):
 root=ABLATION_ROOT/name;pred=root/'predictions';debug=root/'debug';logs=root/'logs'
 for p in [pred,debug,logs]:p.mkdir(parents=True,exist_ok=True)
 cmd=['python','-u','scripts/75_run_round2_local.py','--config',config,'--input-dir','data/raw/input_round_2','--output-dir',str(pred),'--debug-dir',str(debug),'--baseline-prediction-dir','data/baselines/vihealthbert_27_6152','--llama-bin',str(LLAMA_BIN),'--qwen-model','models/gguf/qwen2.5-7b/Qwen2.5-7B-Instruct-Q4_K_M.gguf','--model-alias','qwen2.5-7b','--gpu-layers','99','--threads','4','--server-timeout','600']
 for attempt in range(1,attempts+1):
  if len(list(pred.glob('*.json')))>=100:break
  subprocess.run(['pkill','-f','llama-server.*8081'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
  with (logs/f'attempt_{attempt}.log').open('a',encoding='utf-8',buffering=1) as log:
   p=subprocess.Popen(cmd,cwd=REPO,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
   for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
   p.wait()
  time.sleep(3)
 count=len(list(pred.glob('*.json')));assert count==100,(name,count)
 return root

for name,config in ABLATIONS.items():run_v53(name,config)


In [58]:
### Gliner con
# Chuẩn bị real-dominant positive-chunk mix.
subprocess.run(['python','scripts/108_prepare_v53_real_mix.py','--real-repeat','5','--synthetic-limit','80'],cwd=REPO,check=True,env=env)


{
  "split": "train",
  "source_documents": 530,
  "long_documents": 450,
  "output_chunks": 5940,
  "dropped_empty_chunks": 165,
  "source_entities": 69977,
  "covered_unique_entities": 69977,
  "emitted_entities_with_overlap": 98862,
  "lost_entities": 0,
  "rescue_windows_added": 0,
  "chunk_size": 220,
  "overlap": 64,
  "max_output_tokens": 220,
  "positive_only": true,
  "label_counts_with_overlap": {
    "CHẨN_ĐOÁN": 15747,
    "TRIỆU_CHỨNG": 66920,
    "TÊN_XÉT_NGHIỆM": 9029,
    "KẾT_QUẢ_XÉT_NGHIỆM": 4204,
    "THUỐC": 2962
  }
}
{
  "split": "dev",
  "source_documents": 10,
  "long_documents": 10,
  "output_chunks": 140,
  "dropped_empty_chunks": 0,
  "source_entities": 1557,
  "covered_unique_entities": 1557,
  "emitted_entities_with_overlap": 2206,
  "lost_entities": 0,
  "rescue_windows_added": 0,
  "chunk_size": 220,
  "overlap": 64,
  "max_output_tokens": 220,
  "positive_only": true,
  "label_counts_with_overlap": {
    "CHẨN_ĐOÁN": 365,
    "TRIỆU_CHỨNG": 1511,
    "TÊ

CompletedProcess(args=['python', 'scripts/108_prepare_v53_real_mix.py', '--real-repeat', '5', '--synthetic-limit', '80'], returncode=0)